# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos Industriais

Neste notebook implementamos a biblioteca completa de operadores lógicos (AND, OR, NOT, XOR, IMPLIES, IFF, NAND, NOR) e modelamos os blocos de permissivos de partida (*Start Permissives*) e intertravamento contínuo para os atuadores da fábrica.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from typing import Dict, List, Tuple

def NOT(a: bool) -> bool: return not a
def AND(a: bool, b: bool) -> bool: return a and b
def OR(a: bool, b: bool) -> bool: return a or b
def XOR(a: bool, b: bool) -> bool: return bool(a ^ b)
def IMPLIES(a: bool, b: bool) -> bool: return (not a) or b
def IFF(a: bool, b: bool) -> bool: return a == b

class ControladorIntertravamento:
    @staticmethod
    def permissivo_bomba_P101(l_acid_low: bool, ls_suc_open: bool, p_discharge_high: bool, 
                              e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
        modo_valido = XOR(auto_mode, manual_mode)
        permissivo = (NOT(l_acid_low) and ls_suc_open and NOT(p_discharge_high) and NOT(e1) and modo_valido)
        trip = l_acid_low or NOT(ls_suc_open) or p_discharge_high or e1 or NOT(modo_valido)
        return {
            "Permissivo_Partida": permissivo,
            "Trip_Ativo": trip,
            "Modo_Consistente": modo_valido
        }

cenarios = [
    {"Desc": "1. Operação Normal Auto", "args": (False, True, False, False, True, False)},
    {"Desc": "2. Falha: Nível Baixo", "args": (True, True, False, False, True, False)},
    {"Desc": "3. Falha: Sucção Fechada", "args": (False, False, False, False, True, False)},
    {"Desc": "4. Falha: Sobrepressão", "args": (False, True, True, False, True, False)},
    {"Desc": "5. Falha: Emergência", "args": (False, True, False, True, True, False)},
    {"Desc": "6. Falha: Auto+Manual", "args": (False, True, False, False, True, True)},
]

resultados = []
for c in cenarios:
    res = ControladorIntertravamento.permissivo_bomba_P101(*c["args"])
    resultados.append({
        "Cenário": c["Desc"],
        "Permissivo": "LIBERADO" if res["Permissivo_Partida"] else "BLOQUEADO",
        "Trip": "TRIP ATIVO" if res["Trip_Ativo"] else "SEGURO",
        "Modo OK": res["Modo_Consistente"]
    })

print(formatar_tabela(resultados))
assert resultados[0]["Permissivo"] == "LIBERADO"
assert resultados[1]["Permissivo"] == "BLOQUEADO"
print("\n[OK] Bloco de permissivos validado com sucesso!")


Cenário                  | Permissivo | Trip       | Modo OK
-------------------------+------------+------------+--------
1. Operação Normal Auto  | LIBERADO   | SEGURO     | True   
2. Falha: Nível Baixo    | BLOQUEADO  | TRIP ATIVO | True   
3. Falha: Sucção Fechada | BLOQUEADO  | TRIP ATIVO | True   
4. Falha: Sobrepressão   | BLOQUEADO  | TRIP ATIVO | True   
5. Falha: Emergência     | BLOQUEADO  | TRIP ATIVO | True   
6. Falha: Auto+Manual    | BLOQUEADO  | TRIP ATIVO | False  

[OK] Bloco de permissivos validado com sucesso!
